In [ ]:
import os, sys
from os import listdir
from os.path import join, basename
from datetime import datetime, timezone, timedelta

import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from pytorch_metric_learning import losses
import numpy as np
import numpy.random as npr
from sklearn.model_selection import train_test_split

def elapsed():
    return datetime.now(timezone(timedelta(hours=7))).strftime("%H:%M:%S")

os.makedirs('/kaggle/working/cache', exist_ok=True)

In [ ]:
def scaler(mel):

    log_mel = librosa.power_to_db(mel, ref=np.amax)
    min_, max_ = log_mel.min(), log_mel.max()
    
    if max_ - min_ == 0: return np.zeros_like(log_mel)

    return (log_mel - min_) / (max_ - min_)


def get_sps(y, sr: int):
    '''Перевод музыки в спектограммы'''

    mel_low = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=512,
        n_mels=128, fmin=0, fmax=10_000
    )
    mel_high = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=512,
        n_mels=128, fmin=8_000, fmax=sr//2
    )
    stacked = np.stack(list(map(scaler, (mel_low, mel_high))), axis=0)
    return torch.tensor(stacked, dtype=torch.float32)

In [ ]:
class Data(Dataset):

    def __init__(
            self,
            sr: int = 32_000,
            duration: int = 5,      # секунды
            stride: float = 2.5,    # перекрытие в секундах
            num_segments: int = 5,
            is_train: bool = True
            ) -> None:
        
        self._train_audio = r'/kaggle/input/competitions/birdclef-2026/train_audio'
        self._cache_path = r'/kaggle/working/cache'

        self._sr = sr
        self._num_segments = num_segments
        self._is_train = is_train
    
        self._segment_len = int(duration * sr)
        self._stride = int(stride * sr)

        self._cls2idx = {}
        self._samples = []

        clses = sorted(listdir(self._train_audio))
        for idx,cls in enumerate(clses):
            self._cls2idx[cls] = idx

            cls_path = join(self._train_audio, cls)
            cls_audios = listdir(cls_path)
            for audio in cls_audios:

                self._samples.append((
                    join(cls_path, audio),
                    idx
                ))

        train_samples, valid_samples = train_test_split(
            self._samples,
            test_size=0.2,
            stratify=[s[1] for s in self._samples],
            random_state=42
        )
        self._samples = train_samples if self._is_train else valid_samples


    def _make_slides(self, y, file: str):

        segments = []
        positions = list(range(0, len(y), self._stride))

        if self._is_train:
            is_replace = len(positions) < self._num_segments
            positions = np.sort(npr.choice(positions, self._num_segments, replace=is_replace))
        else:
            if len(positions) > self._num_segments:
                idxs = np.linspace(0, len(positions)-1, self._num_segments).astype(int)
                positions = [positions[i] for i in idxs]
            else:
                positions = npr.choice( positions, self._num_segments, replace=True)

        for start in positions:
            segment = y[start:start+self._segment_len]

            if len(segment) < self._segment_len:
                pad = self._segment_len - len(segment)
                segment = np.pad(segment, (0, pad))

            # кеширование
            full_name = f'{file}_{self._sr}_{self._segment_len}_{start}.pt'
            file_path = join(self._cache_path, full_name)

            if os.path.exists(file_path):
                sps = torch.load(file_path)
            else:
                sps = get_sps(segment, self._sr)
                torch.save(sps, file_path)
            segments.append(sps)

        return torch.stack(segments)
        

    def __len__(self):
        return len(self._samples)
    
    def __getitem__(self, key):
        path, label = self._samples[key]

        y, _ = librosa.load(path, sr=self._sr)
        file = basename(path).split('.')[0]

        return self._make_slides(y, file), torch.tensor(label, dtype=torch.long)

In [ ]:
batch_size = 32
train_dataset = DataLoader(Data(), batch_size=batch_size, num_workers=4, shuffle=True, pin_memory=True)
valid_dataset = DataLoader(Data(is_train=False), batch_size=batch_size, num_workers=4, shuffle=False, pin_memory=True)

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
# for param in model.parameters():
#     param.requires_grad = False

# потому что пока что только 2 спектограммы
model.conv1 = nn.Conv2d(2, 64, 7, 2, 3, bias=False)
model.fc = nn.Identity()
model.cuda()

In [ ]:
criterion = losses.ArcFaceLoss(
    num_classes=206,
    embedding_size=512,
)
optimizer = torch.optim.AdamW(
    params=list(model.parameters()) + list(criterion.parameters()),
    lr=1e-4
)

In [ ]:
green = "\033[92m"
reset = "\033[0m"

EPOCH = 10
total_train, total_valid = len(train_dataset), len(valid_dataset)

best_loss = float('inf')

for epoch in range(EPOCH):

    # train
    model.train()
    train_loss = 0.0
    for xb,yb in train_dataset:
        
        B, N, C, H, W = xb.shape
        x = xb.view(B*N, C, H, W).cuda()
        y = yb.unsqueeze(1).repeat(1, N).view(-1).cuda()

        optimizer.zero_grad()

        emb = F.normalize(model(x))

        loss = criterion(emb, y)
        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= total_train

    # valid
    model.eval()
    valid_acc = 0.0
    with torch.no_grad():
        for xb,yb in valid_dataset:

            B, N, C, H, W = xb.shape
            x = xb.view(B*N, C, H, W).cuda()
            y = yb.unsqueeze(1).repeat(1, N).view(-1).cuda()

            emb = F.normalize(model(x)).view(B, N, -1).max(dim=1)[0]
            W = F.normalize(criterion.W)

            similar = emb @ W.T

            pred = similar.argmax(dim=1)
            acc = (pred == yb.cuda()).float().mean()

            valid_acc += acc.item()

    valid_acc /= total_valid

    if train_loss < best_loss:
        best_loss = train_loss

        torch.save({
            'model': model.state_dict(),
            'arcface': criterion.state_dict()
        }, '/kaggle/working/best_params.pth')
        print(f'Модель {epoch} эпохи сохранена')

    print(f'Epoch: [ {epoch+1:^2} / {EPOCH} ] [{elapsed()}]')
    print(f'  Acc: {green}{valid_acc:.4f}{reset},   TrainLoss: {train_loss:.4f}')